# Station workplace testbench

Interactive testbench for the `station_workplace` lifecycle nodes (`workplace_rfid_scanner` / `workplace_barcode_scanner`), focused on the `~/wait_routing` action.

Prerequisites:
1. Source ROS 2 Jazzy and the workspace overlay, and use a Jupyter kernel from `/opt/venv`.
2. Start the dependencies the workplace node consumes: the product carrier (e.g. the conveyor sandbox from `src/product_carrier/tests/testbench/conveyor`), MoveIt 2 (`/get_planning_scene`, `/apply_planning_scene`), `production_manager`, `object_model_storage` and the RFID or barcode scanner node.
3. Start the workplace node, e.g.:
   ```bash
   ros2 launch station_workplace workplace_rfid_scanner.launch.py
   ```
4. Make sure the kernel uses the same `ROS_DOMAIN_ID` as the launch terminals.

The workplace node starts **Unconfigured**; use the lifecycle cells below to drive it. `~/wait_routing` exists only while the node is **Active**.

## Setup

In [ ]:
import threading
import time

import rclpy
from action_msgs.msg import GoalStatus
from lifecycle_msgs.msg import Transition
from lifecycle_msgs.srv import ChangeState, GetAvailableTransitions, GetState
from rclpy.action import ActionClient
from rclpy.executors import SingleThreadedExecutor
from rclpy.qos import DurabilityPolicy, QoSProfile, ReliabilityPolicy

from station_workplace.action import WaitRouting
from station_workplace.msg import WorkplaceState

TARGET_NODE = '/workplace_rfid_scanner'  # or '/workplace_barcode_scanner'

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('workplace_testbench')
executor = SingleThreadedExecutor()
executor.add_node(node)
spin_thread = threading.Thread(target=executor.spin, daemon=True)
spin_thread.start()

In [ ]:
def call(service_type, service_name, request, timeout=10.0):
    client = node.create_client(service_type, service_name)
    try:
        if not client.wait_for_service(timeout_sec=timeout):
            raise TimeoutError(f'Service {service_name} was not available within {timeout}s')
        future = client.call_async(request)
        deadline = time.monotonic() + timeout
        while not future.done():
            if time.monotonic() > deadline:
                raise TimeoutError(f'Service call {service_name} did not complete within {timeout}s')
            time.sleep(0.01)
        return future.result()
    finally:
        node.destroy_client(client)

def get_state():
    return call(GetState, f'{TARGET_NODE}/get_state', GetState.Request()).current_state.label

def available_transitions():
    response = call(GetAvailableTransitions, f'{TARGET_NODE}/get_available_transitions', GetAvailableTransitions.Request())
    return [t.transition.label for t in response.available_transitions]

def transition(transition_id):
    request = ChangeState.Request()
    request.transition.id = transition_id
    success = call(ChangeState, f'{TARGET_NODE}/change_state', request).success
    print(f'success={success}, state={get_state()}')

## Monitor

Subscribes to the node's latched `~/workplace` topic; changes are printed as they arrive. Inspect `latest` at any time.

In [ ]:
latest = {'workplace': None}

def _on_workplace(msg):
    latest['workplace'] = msg
    print(f'[workplace] in_place={msg.in_place} product_data={list(msg.product_data)}')

latched_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE, durability=DurabilityPolicy.TRANSIENT_LOCAL)
workplace_sub = node.create_subscription(WorkplaceState, f'{TARGET_NODE}/workplace', _on_workplace, latched_qos)

In [ ]:
latest

## Lifecycle: bring up

In [ ]:
print(get_state(), available_transitions())

In [ ]:
transition(Transition.TRANSITION_CONFIGURE)

In [ ]:
transition(Transition.TRANSITION_ACTIVATE)

## wait_routing action (node must be Active)

`send_wait_routing()` sends a goal and returns its goal handle; the result is printed when it arrives (status, `routing_time_s`, `success`, `error_message`, `product_data`). Only one goal runs at a time; a second goal or an invalid `routing_operation` is rejected.

With the simulated conveyor, `inject_product` / `eject_product` are no-ops: after sending the goal, move the pallet in Gazebo (see the conveyor testbench) to trigger the injection / ejection the goal waits for.

In [ ]:
STATUS = {GoalStatus.STATUS_SUCCEEDED: 'SUCCEEDED', GoalStatus.STATUS_ABORTED: 'ABORTED',
          GoalStatus.STATUS_CANCELED: 'CANCELED'}
wait_routing_client = ActionClient(node, WaitRouting, f'{TARGET_NODE}/wait_routing')

def _print_result(future):
    response = future.result()
    result = response.result
    print(f'[wait_routing] status={STATUS.get(response.status, response.status)} '
          f'routing_time_s={result.routing_time_s} success={result.success} '
          f'error_message={result.error_message!r} product_data={list(result.product_data)}')

def send_wait_routing(routing_operation, timeout=5.0):
    if not wait_routing_client.wait_for_server(timeout_sec=timeout):
        raise TimeoutError(f'{TARGET_NODE}/wait_routing not available (is the node active?)')
    future = wait_routing_client.send_goal_async(WaitRouting.Goal(routing_operation=routing_operation))
    deadline = time.monotonic() + timeout
    while not future.done():
        if time.monotonic() > deadline:
            raise TimeoutError('Goal response not received')
        time.sleep(0.01)
    goal_handle = future.result()
    print(f'[wait_routing] {routing_operation}: accepted={goal_handle.accepted}')
    if goal_handle.accepted:
        goal_handle.get_result_async().add_done_callback(_print_result)
    return goal_handle

In [ ]:
goal = send_wait_routing(WaitRouting.Goal.ROUTING_OPERATION_INJECTION)

In [ ]:
goal = send_wait_routing(WaitRouting.Goal.ROUTING_OPERATION_EJECTION)

In [ ]:
# Cancel the last goal (only stops the wait; the carrier operation is not affected)
goal.cancel_goal_async()

In [ ]:
# Rejected: invalid routing_operation
send_wait_routing('teleport')

## Lifecycle: bring down

Deactivating while a goal is running aborts it (`wait_routing aborted: node deactivated`).

In [ ]:
transition(Transition.TRANSITION_DEACTIVATE)

In [ ]:
transition(Transition.TRANSITION_CLEANUP)

In [ ]:
# Final: the node cannot be brought back without restarting it
# transition(Transition.TRANSITION_UNCONFIGURED_SHUTDOWN)

## Teardown

In [ ]:
wait_routing_client.destroy()
node.destroy_subscription(workplace_sub)
executor.shutdown()
spin_thread.join(timeout=5.0)
node.destroy_node()
rclpy.shutdown()